# Tutorial 1: What Is an Optimization Problem?

**Prerequisites:** None  
**Up Next:** Tutorial 2 — Types of Optimization Problems

---

## Concept

An optimization problem asks: *given a set of choices, which one is best?* Formally, we seek values of **design variables** $\mathbf{x}$ that minimize an **objective function** $f(\mathbf{x})$, possibly subject to **constraints** $g(\mathbf{x}) \le 0$. The set of $\mathbf{x}$ satisfying all constraints is called the **feasible region**.

## Key Takeaway

> **Every optimization problem has three ingredients: design variables (what you can change), an objective (what you want to minimize), and constraints (what you must satisfy).**

## Math Core

$$\min_{\mathbf{x}} \; f(\mathbf{x}) \quad \text{subject to} \quad g_i(\mathbf{x}) \le 0, \; i = 1, \dots, m$$

where $\mathbf{x} \in \mathbb{R}^n$ are the design variables, $f : \mathbb{R}^n \to \mathbb{R}$ is the objective function, and $g_i$ are inequality constraints.

## Code

We will use a real engineering problem as our running example throughout this series: **synthesizing a four-bar linkage** so that a coupler point traces a desired path.

The [`dms`](https://github.com/JonathanCamargo/dynamics_of_mechanical_systems) library handles mechanism kinematics so we can focus on optimization.

In [ ]:
from dms.mechanisms.fourbar import FourBar
from dms.curves import CompareCurves
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Fixed link lengths
L0, L1 = 1.0, 1.0
# Coupler point offset (along and perpendicular to coupler bar)
PX, PY = 0.5, 0.3
# Target path: 6 points the coupler should pass through
TARGETS = np.array([
    [1.0, 1.5], [0.75, 1.5], [0.5, 1.5],
    [0.25, 1.5], [0.0, 1.5], [-0.25, 1.5]
])
# Crank angles to evaluate
THETAS = np.linspace(np.deg2rad(60), np.deg2rad(120), len(TARGETS))

### The running example

A **four-bar linkage** has four rigid links connected by revolute joints.  We fix the ground link ($\ell_0 = 1$) and the crank ($\ell_1 = 1$), and treat the coupler length $\ell_2$ and follower length $\ell_3$ as our two **design variables**: $\mathbf{x} = [\ell_2, \ell_3]$.

As the crank rotates, a point on the coupler bar traces a curve.  Our goal is to find $\ell_2$ and $\ell_3$ so that this coupler path passes as close as possible to a set of **target points**.

In [ ]:
def coupler_path(l2, l3, n_pts=6):
    """Sweep the crank and return the coupler point trajectory."""
    mechanism = FourBar(1, 1, l2, l3)
    thetas = np.linspace(np.deg2rad(60), np.deg2rad(120), n_pts)
    path = []
    for theta1 in thetas:
        z, fk = mechanism.FK(theta1)
        if fk.cost > 1e-3:       # mechanism cannot assemble
            return None
        pts = mechanism.ComputePoints(theta1, *z)
        A, B = pts['A'], pts['B']
        bar = B - A
        u = bar / np.linalg.norm(bar)
        perp = np.array([-u[1], u[0]])
        path.append(A + px * u + py * perp)
    return np.array(path)

In [ ]:
def fourbar_fk(theta1, l0, l1, l2, l3):
    """Solve four-bar loop closure for passive angles (theta2, theta3).
    
    Returns (theta2, theta3) or None if the mechanism can't assemble.
    Uses the cosine-law closed-form solution.
    """
    # Position of crank tip A
    ax, ay = l1 * np.cos(theta1), l1 * np.sin(theta1)
    # Vector from ground pivot D (at l0, 0) to A
    dx, dy = ax - l0, ay
    d = np.sqrt(dx**2 + dy**2)
    # Triangle D-A-B: sides l3, d, l2
    cos_alpha = (l3**2 + d**2 - l2**2) / (2 * l3 * d)
    if np.abs(cos_alpha) > 1:
        return None  # can't assemble
    alpha = np.arccos(np.clip(cos_alpha, -1, 1))
    beta = np.arctan2(dy, dx)
    theta3 = beta + alpha  # elbow-up branch
    # B position from A
    bx = l0 + l3 * np.cos(theta3)
    by = l3 * np.sin(theta3)
    theta2 = np.arctan2(by - ay, bx - ax)
    return theta2, theta3


def coupler_point(theta1, l2, l3):
    """Compute coupler point position for given crank angle and link lengths."""
    sol = fourbar_fk(theta1, L0, L1, l2, l3)
    if sol is None:
        return None
    theta2, _ = sol
    ax, ay = L1 * np.cos(theta1), L1 * np.sin(theta1)
    # Unit vectors along and perpendicular to coupler bar
    ux, uy = np.cos(theta2), np.sin(theta2)
    return np.array([ax + PX * ux - PY * uy,
                     ay + PX * uy + PY * ux])


def objective(x):
    """Objective f(x): how far is the coupler path from the targets?"""
    l2, l3 = x
    path = []
    for theta1 in THETAS:
        pt = coupler_point(theta1, l2, l3)
        if pt is None:
            return 1e3  # infeasible penalty
        path.append(pt)
    path = np.array(path)
    return CompareCurves(path[:, 0], path[:, 1],
                         TARGETS[:, 0], TARGETS[:, 1])

In [ ]:
l2_vals = np.linspace(0.3, 2.5, 40)
l3_vals = np.linspace(0.3, 2.5, 40)
L2, L3 = np.meshgrid(l2_vals, l3_vals)
Z = np.vectorize(lambda a, b: objective([a, b]))(L2, L3)

fig, ax = plt.subplots(figsize=(7, 5))
cf = ax.contourf(L2, L3, np.log1p(Z), levels=30, cmap='viridis')
ax.set_xlabel(r'$\ell_2$ (coupler length)')
ax.set_ylabel(r'$\ell_3$ (follower length)')
ax.set_title('Objective landscape: four-bar path synthesis')
plt.colorbar(cf, label=r'$\ln(1 + f)$')
plt.tight_layout()

### Visualizing a candidate mechanism

Let's pick a guess and see how the linkage moves.

In [ ]:
l2_vals = np.linspace(0.3, 2.5, 80)
l3_vals = np.linspace(0.3, 2.5, 80)
L2g, L3g = np.meshgrid(l2_vals, l3_vals)
Z = np.vectorize(lambda a, b: objective([a, b]))(L2g, L3g)

fig, ax = plt.subplots(figsize=(7, 5))
cf = ax.contourf(L2g, L3g, np.log1p(Z), levels=30, cmap='viridis')
ax.plot(*TARGETS.T, 'r*', ms=8, zorder=5)
ax.set_xlabel(r'$\ell_2$ (coupler length)')
ax.set_ylabel(r'$\ell_3$ (follower length)')
ax.set_title('Objective landscape: four-bar path synthesis')
plt.colorbar(cf, label=r'$\ln(1 + f)$')
plt.tight_layout()

The coupler path doesn't match the targets very well. The rest of this tutorial series teaches you systematic methods to find the best $\ell_2$ and $\ell_3$ — from gradient descent to genetic algorithms to hybrid strategies.

---

l2, l3 = 1.0, 1.5
mechanism = FourBar(L0, L1, l2, l3)

# Compute coupler path using our fast FK
path = np.array([coupler_point(t, l2, l3) for t in THETAS])

fig, ax = plt.subplots(figsize=(6, 5))
mechanism.plot(np.deg2rad(90), ax=ax)
ax.plot(TARGETS[:, 0], TARGETS[:, 1], 'r*', ms=10, label='targets')
ax.plot(path[:, 0], path[:, 1], 'b.-', label='coupler path')
ax.set_aspect('equal')
ax.legend()
ax.set_title(f'Four-bar: $\\ell_2$={l2}, $\\ell_3$={l3}  |  f = {objective([l2, l3]):.3f}')
plt.tight_layout()